# Learning Deep Learning using PyTorch

Experiments with ResNet-18 inference, residual-block and activation ablations, and CIFAR-10 training preparation.

**Status:** Inference and forward-pass comparisons are demonstrated. The CIFAR-10 training loop and measured loss curves are not yet implemented.

In [ ]:
import torch
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = resnet18(weights=ResNet18_Weights.DEFAULT).to(device)
model.eval()

x = torch.randn(4, 3, 224, 224, device=device)
with torch.inference_mode():
    output = model(x)

print(output.shape)  # torch.Size([4, 1000])
print(output)

## Identity versus ReLU

`nn.Identity()` returns its input unchanged. `nn.ReLU()` sets negative values to zero.

In [ ]:
import torch
import torch.nn as nn

neuron1 = nn.Identity()
neuron2 = nn.ReLU()
x = torch.tensor([1.0, 2.0, 3.0])
y = neuron1(x) + neuron2(x)

print(y)  # tensor([2., 4., 6.])

## Residual-block ablation

Compare the pretrained network against a copy with the full `layer2[1]` block replaced by `nn.Identity()`. This measures a changed forward pass; it does not measure training quality.

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(4, 3, 224, 224, device=device)

model1 = resnet18(weights=ResNet18_Weights.DEFAULT)
model1.layer2[1] = nn.Identity()
model1 = model1.to(device).eval()

model2 = resnet18(weights=ResNet18_Weights.DEFAULT).to(device).eval()

with torch.inference_mode():
    output1 = model1(x)
    output2 = model2(x)

print("Ablated output:", output1)
print("Original output:", output2)

## Activation-function ablation

Keep the pretrained weights and input fixed, and compare ReLU, LeakyReLU, and GELU. In a ResNet `BasicBlock`, the same `relu` module is called more than once, so replacing `layer2[1].relu` changes both invocations. These are output comparisons, **not** a fair training-performance ranking.

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(4, 3, 224, 224, device=device)

activations = {
    "ReLU": nn.ReLU,
    "LeakyReLU": nn.LeakyReLU,
    "GELU": nn.GELU,
}
results = {}

with torch.inference_mode():
    for name, activation in activations.items():
        model = resnet18(weights=ResNet18_Weights.DEFAULT)
        model.layer2[1].relu = activation()
        model = model.to(device).eval()
        results[name] = model(x).cpu()
        print(f"{name}: {results[name].shape}")

print("ReLU versus GELU equal?", torch.allclose(results["ReLU"], results["GELU"]))

## Continual learning: catastrophic forgetting

Three common families of approaches to mitigating catastrophic forgetting:

- Regularization
- Replay
- Parameter isolation

These are study notes, not implemented experiments in this notebook.

# Training ResNet-18 on CIFAR-10 (in progress)

The following cells prepare reproducibility controls, transforms, loaders, and the initial CIFAR-sized model. Training/validation loops and empirical results still need to be added.

In [ ]:
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torchvision.models import resnet18
from torch.utils.data import DataLoader, random_split

## Set up random seeds and device

Set Python, NumPy, and PyTorch random seeds before initializing models. Seed the dataset split independently for repeatable train/validation partitions.

A seed improves reproducibility but does not by itself guarantee deterministic GPU operations.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
EPOCHS = 10
BATCH_SIZE = 128

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## Prepare dataset

- `transforms.ToTensor()` converts input images to float tensors in [0, 1].
- `transforms.Normalize(mean, std)` standardizes each image channel.
- `datasets.CIFAR10(train=True)` provides 50,000 training images.
- `random_split(..., generator=...)` creates a repeatable 45,000/5,000 train/validation split.
- `DataLoader(..., shuffle=True)` shuffles training order each epoch; validation uses `shuffle=False`.

References: [ToTensor](https://pytorch.org/vision/stable/generated/torchvision.transforms.ToTensor.html), [Normalize](https://pytorch.org/vision/stable/generated/torchvision.transforms.Normalize.html), [CIFAR10](https://pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html), [random_split](https://pytorch.org/docs/stable/data.html#torch.utils.data.random_split).

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616),
    ),
])

dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

train_set, val_set = random_split(
    dataset,
    [45000, 5000],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

print(len(train_set), len(val_set))  # 45000 5000

## Prepare model

`num_classes=10` matches CIFAR-10's ten image classes. For its small 32×32 images, replace the usual 7×7 stride-2 input convolution with a 3×3 stride-1 convolution and remove the initial max-pool.

This creates a model from scratch (`weights=None`), **not** a fine-tuned ImageNet model.

In [ ]:
# Shared initial model for future controlled training ablations
base_model = resnet18(weights=None, num_classes=10)

# Adapt ResNet-18 to 32×32 CIFAR-10 inputs
base_model.conv1 = nn.Conv2d(
    3, 64, kernel_size=3, stride=1, padding=1, bias=False
)
base_model.maxpool = nn.Identity()

print(base_model.fc.out_features)  # 10

# TODO: clone this initialization for each activation condition,
# create criterion and optimizer, and add train/validation loops.
# Compare loss curves across identical splits, seeds, and schedules.